In [1]:

# 1. Verificar ambiente (Colab vs Local) e Instalar Dependências
import os
import sys

try:
    from google.colab import drive
    IN_COLAB = True
    print("Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("Running locally")

if IN_COLAB:
    # Clone repo if not exists
    if not os.path.exists('/content/ufc-easytpp'):
        !git clone https://github.com/hugoramos/ufc-easytpp.git /content/ufc-easytpp
    
    # Install dependencies
    !pip install -q torch numpy matplotlib pandas datasets pyyaml transformers

    # Set working directory
    %cd /content/ufc-easytpp
    
    # Add to sys.path
    if '/content/ufc-easytpp' not in sys.path:
        sys.path.append('/content/ufc-easytpp')

else:
    # Local setup
    local_path = '/Users/hugoramossoares/Sites/EasyTemporalPointProcess'
    if os.path.exists(local_path):
        os.chdir(local_path)
        if local_path not in sys.path:
            sys.path.append(local_path)
        print(f"Working directory set to: {os.getcwd()}")
    else:
        print(f"Warning: Local path {local_path} not found.")

# Verify imports
import torch
import numpy as np
import matplotlib.pyplot as plt
from easy_tpp.model.torch_model.torch_nhp import NHP
from easy_tpp.model.torch_model.torch_thp import THP
# from easy_tpp.model.torch_model.torch_rothp import RoTHP # Uncomment if RoTHP is available
print("Bibliotecas EasyTPP carregadas com sucesso.")


Running locally
Working directory set to: /Users/hugoramossoares/Sites/EasyTemporalPointProcess
Bibliotecas EasyTPP carregadas com sucesso.


In [2]:

# 1. Verificar ambiente (Colab vs Local) e Instalar Dependências
import os
import sys

try:
    from google.colab import drive
    IN_COLAB = True
    print("Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("Running locally")

if IN_COLAB:
    # Clone repo if not exists
    if not os.path.exists('/content/ufc-easytpp'):
        !git clone https://github.com/hugoramos/ufc-easytpp.git /content/ufc-easytpp
    
    # Install dependencies
    !pip install -q torch numpy matplotlib pandas datasets pyyaml transformers

    # Set working directory
    %cd /content/ufc-easytpp
    
    # Add to sys.path
    if '/content/ufc-easytpp' not in sys.path:
        sys.path.append('/content/ufc-easytpp')

else:
    # Local setup
    local_path = '/Users/hugoramossoares/Sites/EasyTemporalPointProcess'
    if os.path.exists(local_path):
        os.chdir(local_path)
        if local_path not in sys.path:
            sys.path.append(local_path)
        print(f"Working directory set to: {os.getcwd()}")
    else:
        print(f"Warning: Local path {local_path} not found.")

# Verify imports
import torch
import numpy as np
import matplotlib.pyplot as plt
from easy_tpp.model.torch_model.torch_nhp import NHP
from easy_tpp.model.torch_model.torch_thp import THP
# from easy_tpp.model.torch_model.torch_rothp import RoTHP # Uncomment if RoTHP is available
print("Bibliotecas EasyTPP carregadas com sucesso.")


Running locally
Working directory set to: /Users/hugoramossoares/Sites/EasyTemporalPointProcess
Bibliotecas EasyTPP carregadas com sucesso.


In [3]:

# 2. Baixar Datasets (MIMIC-II e Financial)
# Fonte: SimiaoZuo/Transformer-Hawkes-Process (Github Mirror)

import os
import pickle
import urllib.request

DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True)

datasets_info = {
    'mimic': {
        'train': 'https://github.com/SimiaoZuo/Transformer-Hawkes-Process/raw/master/data/mimic/train.pkl',
        'dev': 'https://github.com/SimiaoZuo/Transformer-Hawkes-Process/raw/master/data/mimic/dev.pkl',
        'test': 'https://github.com/SimiaoZuo/Transformer-Hawkes-Process/raw/master/data/mimic/test.pkl'
    },
    'financial': {
        'train': 'https://github.com/SimiaoZuo/Transformer-Hawkes-Process/raw/master/data/financial/train.pkl',
        'dev': 'https://github.com/SimiaoZuo/Transformer-Hawkes-Process/raw/master/data/financial/dev.pkl',
        'test': 'https://github.com/SimiaoZuo/Transformer-Hawkes-Process/raw/master/data/financial/test.pkl'
    }
}

def download_dataset(name):
    print(f"\nBaixando dataset: {name}...")
    target_dir = os.path.join(DATA_DIR, name)
    os.makedirs(target_dir, exist_ok=True)
    
    for split, url in datasets_info[name].items():
        file_path = os.path.join(target_dir, f'{split}.pkl')
        if not os.path.exists(file_path):
            print(f"  Downloading {split}.pkl from {url}...")
            try:
                # Use wget command for robustness in notebooks
                !wget -q -O {file_path} {url}
                if os.path.getsize(file_path) < 1000: # Check for 404 or small error file
                     print(f"    Warning: File {file_path} seems too small. Checking content...")
            except Exception as e:
                print(f"    Error downloading {split}: {e}")
        else:
            print(f"  {split}.pkl already exists.")

download_dataset('mimic')
download_dataset('financial')

print("\nVerificando arquivos baixados:")
!ls -R data/mimic data/financial



Baixando dataset: mimic...
zsh:1: command not found: wget
    Error downloading train: [Errno 2] No such file or directory: 'data/mimic/train.pkl'
zsh:1: command not found: wget
    Error downloading dev: [Errno 2] No such file or directory: 'data/mimic/dev.pkl'
zsh:1: command not found: wget
    Error downloading test: [Errno 2] No such file or directory: 'data/mimic/test.pkl'

Baixando dataset: financial...
zsh:1: command not found: wget
    Error downloading train: [Errno 2] No such file or directory: 'data/financial/train.pkl'
zsh:1: command not found: wget
    Error downloading dev: [Errno 2] No such file or directory: 'data/financial/dev.pkl'
zsh:1: command not found: wget
    Error downloading test: [Errno 2] No such file or directory: 'data/financial/test.pkl'

Verificando arquivos baixados:
data/financial:

data/mimic:


In [ ]:

# 3. Carregar e Preprocessar Dados

def load_data(dataset_name):
    base_path = os.path.join(DATA_DIR, dataset_name)
    data = {}
    for split in ['train', 'dev', 'test']:
        path = os.path.join(base_path, f'{split}.pkl')
        try:
            with open(path, 'rb') as f:
                # Handle potential python 2/3 pickle issues
                data[split] = pickle.load(f, encoding='latin1')
        except Exception as e:
            print(f"Erro ao carregar {path}: {e}")
            return None
            
    # Standardize format to list of dicts if necessary
    # Check structure
    sample = data['train'][0]
    print(f"\nEstrutura de {dataset_name} (exemplo):")
    print(sample.keys() if isinstance(sample, dict) else "Not a dict")
    
    return data

mimic_data = load_data('mimic')
financial_data = load_data('financial')

def get_stats(data, name):
    if not data: return None, None
    train = data['train']
    lens = [len(x['time_since_start']) for x in train]
    
    num_types = 0
    all_deltas = []
    for x in train:
        num_types = max(num_types, max(x['type_event']))
        all_deltas.extend([d for d in x['time_since_last_event'] if d > 0])
        
    num_types += 1
    time_scale = np.mean(all_deltas)
    
    print(f"\nEstatísticas {name}:")
    print(f"  Sequências Treino: {len(train)}")
    print(f"  Comprimento Médio: {np.mean(lens):.1f}")
    print(f"  Num Event Types:   {num_types}")
    print(f"  Time Scale (Mean Delta): {time_scale:.4f}")
    
    return num_types, time_scale

MIMIC_TYPES, MIMIC_SCALE = get_stats(mimic_data, 'MIMIC-II')
FINANCIAL_TYPES, FINANCIAL_SCALE = get_stats(financial_data, 'Financial')


In [ ]:

# 4. Configuração do Modelo e Funções Auxiliares

# Função Collate para Batching com Normalização Temporal
def collate_fn_factory(time_scale):
    def collate_fn(batch_list):
        time_seqs = []
        time_delta_seqs = []
        type_seqs = []
        
        max_len = 0
        for item in batch_list:
            ts = item['time_since_start']
            td = item['time_since_last_event']
            ev = item['type_event']
            if len(ts) > max_len: max_len = len(ts)
            
            # NORMALIZAÇÃO: Dividir tempos pela escala temporal média
            # Isso ajuda modelos como THP que usam embeddings posicionais baseados em tempo
            ts_norm = torch.tensor(ts, dtype=torch.float64)
            ts_norm = (ts_norm - ts_norm[0]) / time_scale # Zero-start and scale
            
            td_norm = torch.tensor(td, dtype=torch.float64) / time_scale
            
            time_seqs.append(ts_norm.float())
            time_delta_seqs.append(td_norm.float())
            type_seqs.append(torch.tensor(ev, dtype=torch.long))
    
        batch_size = len(batch_list)
        pad_time = torch.zeros(batch_size, max_len)
        pad_delta = torch.zeros(batch_size, max_len)
        pad_type = torch.zeros(batch_size, max_len, dtype=torch.long)
        attention_mask = torch.zeros(batch_size, max_len, max_len)
        batch_non_pad_mask = torch.zeros(batch_size, max_len)
        
        for i in range(batch_size):
            l = len(time_seqs[i])
            pad_time[i, :l] = time_seqs[i]
            pad_delta[i, :l] = time_delta_seqs[i]
            pad_type[i, :l] = type_seqs[i]
            batch_non_pad_mask[i, :l] = 1
            
            # Causal mask + Padding Mask
            causal_mask = torch.triu(torch.ones(max_len, max_len), diagonal=1)
            causal_mask[:, l:] = 1 # Pad cols
            causal_mask[l:, :] = 1 # Pad rows
            attention_mask[i] = causal_mask
    
        return (pad_time, pad_delta, pad_type, batch_non_pad_mask, attention_mask)
    return collate_fn

class ModelConfig:
    def __init__(self, num_types, pad_id, hidden_size=64, num_heads=4, num_layers=2):
        self.num_event_types = num_types
        self.num_event_types_pad = num_types + 1 # Pad is usually last
        self.pad_token_id = num_types
        self.hidden_size = hidden_size
        self.time_emb_size = hidden_size
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.dropout_rate = 0.1
        self.use_ln = True
        self.gpu = 0 if torch.cuda.is_available() else -1
        
        # Thinning for prediction
        self.thinning = type('ThinningConfig', (), {
            'num_sample': 100, 'num_exp': 500, 'over_sample_rate': 10.0, 
            'patience_counter': 5, 'num_samples_boundary': 20, 'dtime_max': 5.0
        })()
        
        # NHP Specifics
        self.model_specs = {'beta': 1.0, 'bias': True}

def compute_metrics(model, test_ds, collate_fn):
    model.eval()
    total_acc, total_rmse, total_events = 0, 0, 0
    subset_size = min(200, len(test_ds))
    subset = test_ds[:subset_size] 
    
    with torch.no_grad():
        for i in range(0, len(subset), 32):
            batch = collate_fn(subset[i:i+32])
            _, time_delta_target, type_target, mask_target, _ = batch
            
            # Predict
            dtimes_pred, types_pred = model.predict_one_step_at_every_event(batch)
            
            # Align targets (ignore first event for prediction validation)
            target_types = type_target[:, 1:]
            target_deltas = time_delta_target[:, 1:]
            target_mask = mask_target[:, 1:]
            
            # Metrics
            correct = (types_pred == target_types) * target_mask
            se = ((dtimes_pred - target_deltas) ** 2) * target_mask
            
            total_acc += correct.sum().item()
            total_rmse += se.sum().item()
            total_events += target_mask.sum().item()
            
    return total_acc / (total_events+1e-9), np.sqrt(total_rmse / (total_events+1e-9))


In [ ]:

# 5. Loop de Treinamento

import time

def train_eval_loop(model_class, name, config, train_ds, test_ds, time_scale, epochs=20):
    print(f"\n>>> Treinando: {name}")
    model = model_class(config)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    collate_fn = collate_fn_factory(time_scale)
    
    history_nll = []
    
    start_time = time.time()
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        total_events = 0
        
        # Train on subset of batches per epoch to speed up demo if needed
        # Use full dataset or subset
        indices = np.random.permutation(len(train_ds))
        if len(indices) > 1000: indices = indices[:1000]
        
        for i in range(0, len(indices), 64):
            batch_list = [train_ds[k] for k in indices[i:i+64]]
            batch = collate_fn(batch_list)
            
            optimizer.zero_grad()
            loss, num_events = model.loglike_loss(batch)
            
            if torch.isnan(loss):
                print("Loss is NaN!")
                continue
                
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            total_loss += loss.item()
            total_events += num_events
            
        train_nll = total_loss / (total_events + 1e-9)
        
        # Eval
        model.eval()
        val_loss, val_events = 0, 0
        with torch.no_grad():
            subset = test_ds[:100]
            batch = collate_fn(subset)
            vl, vn = model.loglike_loss(batch)
            val_loss += vl.item()
            val_events += vn
        val_nll = val_loss / (val_events + 1e-9)
        
        print(f"  Ep {epoch+1}/{epochs} | Train NLL: {train_nll:.4f} | Val NLL: {val_nll:.4f}")
        history_nll.append(val_nll)
        
    total_time = time.time() - start_time
    
    # Final Metrics
    print(f"  Calculando métricas finais (Acc, RMSE)...")
    acc, rmse = compute_metrics(model, test_ds, collate_fn)
    
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"  >>> Resultado Final {name}: Acc={acc:.4f}, RMSE={rmse:.4f}")
    print(f"  >>> Tempo: {total_time:.1f}s | Parâmetros: {num_params:,}")
    
    return {'nll': history_nll, 'acc': acc, 'rmse': rmse, 'time': total_time, 'params': num_params, 'model': model}


In [ ]:

# 6. Experimento 1: MIMIC-II
print("="*40)
print("EXPERIMENTO MIMIC-II")
print("="*40)

if MIMIC_TYPES:
    mimic_config = ModelConfig(num_types=MIMIC_TYPES, pad_id=MIMIC_TYPES, hidden_size=64)
    
    # Treinar NHP
    res_mimic_nhp = train_eval_loop(NHP, "NHP (MIMIC)", mimic_config, mimic_data['train'], mimic_data['test'], MIMIC_SCALE, epochs=20)
    
    # Treinar THP
    res_mimic_thp = train_eval_loop(THP, "THP (MIMIC)", mimic_config, mimic_data['train'], mimic_data['test'], MIMIC_SCALE, epochs=20)
    
    # Plot Comparison
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(res_mimic_nhp['nll'], label='NHP')
    plt.plot(res_mimic_thp['nll'], label='THP')
    plt.title('MIMIC-II: NLL Convergence')
    plt.xlabel('Epochs'); plt.ylabel('NLL')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    metrics = ['Acc', 'RMSE']
    nhp_vals = [res_mimic_nhp['acc'], res_mimic_nhp['rmse']]
    thp_vals = [res_mimic_thp['acc'], res_mimic_thp['rmse']]
    x = np.arange(len(metrics))
    plt.bar(x - 0.2, nhp_vals, 0.4, label='NHP')
    plt.bar(x + 0.2, thp_vals, 0.4, label='THP')
    plt.xticks(x, metrics)
    plt.title('MIMIC-II: Acc & RMSE')
    plt.legend()
    plt.show()


In [ ]:

# 7. Experimento 2: Financial
print("="*40)
print("EXPERIMENTO FINANCIAL")
print("="*40)

if FINANCIAL_TYPES:
    fin_config = ModelConfig(num_types=FINANCIAL_TYPES, pad_id=FINANCIAL_TYPES, hidden_size=64)
    
    res_fin_nhp = train_eval_loop(NHP, "NHP (Financial)", fin_config, financial_data['train'], financial_data['test'], FINANCIAL_SCALE, epochs=20)
    res_fin_thp = train_eval_loop(THP, "THP (Financial)", fin_config, financial_data['train'], financial_data['test'], FINANCIAL_SCALE, epochs=20)
    
    # Plot Comparison
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(res_fin_nhp['nll'], label='NHP')
    plt.plot(res_fin_thp['nll'], label='THP')
    plt.title('Financial: NLL Convergence')
    plt.xlabel('Epochs'); plt.ylabel('NLL')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    metrics = ['Acc', 'RMSE']
    nhp_vals = [res_fin_nhp['acc'], res_fin_nhp['rmse']]
    thp_vals = [res_fin_thp['acc'], res_fin_thp['rmse']]
    x = np.arange(len(metrics))
    plt.bar(x - 0.2, nhp_vals, 0.4, label='NHP')
    plt.bar(x + 0.2, thp_vals, 0.4, label='THP')
    plt.xticks(x, metrics)
    plt.title('Financial: Acc & RMSE')
    plt.legend()
    plt.show()


In [ ]:

# 8. Visualização da Função de Intensidade
# Comparar como NHP e THP modelam a intensidade de um evento específico

def plot_intensity_comparison(models_dict, dataset, time_scale, seq_idx=0, interval_idx=5):
    seq = dataset[seq_idx]
    
    # Preparar dados do intervalo
    ts = torch.tensor(seq['time_since_start'], dtype=torch.float64)
    ts = (ts - ts[0]) / time_scale
    ts = ts.float()
    
    td = torch.tensor(seq['time_since_last_event'], dtype=torch.float64) / time_scale
    td = td.float()
    
    ev = torch.tensor(seq['type_event'], dtype=torch.long)
    
    # Intervalo de interesse: entre evento [interval_idx] e [interval_idx+1]
    t_start = ts[interval_idx].item()
    t_end = ts[interval_idx+1].item()
    
    if t_end <= t_start: return
    
    # Grid de tempo para plotar
    t_grid = torch.linspace(t_start, t_end, 100)
    dt_grid = t_grid - t_start # delta t relativo ao último evento
    
    # Batch único até o evento anterior
    batch_seq_len = interval_idx + 1
    
    # Loop modelos
    plt.figure(figsize=(10, 5))
    
    for name, model in models_dict.items():
        model.eval()
        
        # Preparar batch para compute_intensities
        # Input shape: [1, seq_len]
        time_seq = ts[:batch_seq_len].unsqueeze(0)
        td_seq = td[:batch_seq_len].unsqueeze(0)
        type_seq = ev[:batch_seq_len].unsqueeze(0)
        
        # Sample times: [1, 1, num_samples] -> broadcasting requires careful shape
        # O método espera sample_dtimes com shape [batch, seq_len, num_samples]
        # Queremos avaliar APENAS para o último passo (interval_idx)
        
        num_samples = len(dt_grid)
        sample_dtimes = torch.zeros(1, batch_seq_len, num_samples)
        sample_dtimes[0, -1, :] = dt_grid # Preencher apenas o último passo
        
        with torch.no_grad():
            # Intensities: [batch, seq_len, num_samples, num_types]
            intensities = model.compute_intensities_at_sample_times(
                time_seq, td_seq, type_seq, sample_dtimes, compute_last_step_only=True
            )
            
            # Pegar intensidade total (soma dos tipos) no último passo
            # intensities shape after optim: [1, 1, num_samples, num_types]
            lambda_t = intensities[0, 0, :, :].sum(dim=-1).numpy()
            
        plt.plot(t_grid.numpy(), lambda_t, label=name)
        
    plt.axvline(x=t_end, color='black', linestyle='--', label='Próximo Evento')
    plt.xlabel('Tempo Normalizado')
    plt.ylabel('Intensidade λ(t)')
    plt.title(f'Função de Intensidade: Sequência {seq_idx}, Evento {interval_idx}->{interval_idx+1}')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

print("Gerando gráfico de intensidade para MIMIC...")
if MIMIC_TYPES:
    models = {'NHP': res_mimic_nhp['model'], 'THP': res_mimic_thp['model']}
    plot_intensity_comparison(models, mimic_data['test'], MIMIC_SCALE, seq_idx=0, interval_idx=5)



# 9. Investigação: Por que o THP pode não superar o NHP?

Se os resultados acima mostrarem que o THP (Transformer) tem desempenho similar ou inferior ao NHP (RNN), considere os seguintes fatores críticos implementados neste notebook:

## 1. Normalização Temporal (Crucial para THP)
O THP utiliza embeddings posicionais baseados no tempo (`Time2Vec` ou sinusoidal). Se os tempos (`time_since_start`) forem muito grandes (ex: 1000, 10000), os embeddings oscilam muito rapidamente ou saturam, dificultando o aprendizado.
*   **Solução Aplicada:** Neste notebook, implementamos a divisão por `TIME_SCALE` (média dos deltas) no `collate_fn`. Isso traz os tempos para uma escala próxima de 1.0, facilitando a convergência do THP. Sem isso, o NHP (que usa LSTM e gates) costuma ser mais robusto a escalas variadas.

## 2. Função de Intensidade e Decaimento
O THP original assume um kernel de influência constante ou complexo via self-attention, mas pode ter dificuldade em modelar o decaimento exponencial clássico de Hawkes se não tiver um bias indutivo forte.
*   **Análise:** Observe os gráficos de intensidade (Cell 8). Se o THP produz uma linha reta ou muito ruidosa enquanto o NHP produz o decaimento suave esperado, isso indica que o mecanismo de atenção não aprendeu a dependência temporal correta.

## 3. Tamanho do Dataset e Overfitting
Transformers (THP) geralmente requerem mais dados para generalizar do que RNNs (NHP).
*   **Datasets:** MIMIC-II e Financial são relativamente pequenos/médios. O NHP pode ser mais eficiente em termos de amostras.

## 4. Hiperparâmetros
*   **Hidden Size:** Usamos 64. O paper original pode ter usado 128 ou 256.
*   **Layers:** Usamos 2 camadas. Transformers profundos podem ser difíceis de treinar sem warm-up de LR adequado.

**Conclusão:** A normalização temporal é o fator #1 para o sucesso do THP. Se já estiver aplicada e o desempenho continuar baixo, sugere-se aumentar o `hidden_size` ou usar uma variante como `THP-ExpDecay` (que força um decaimento exponencial na atenção).
